# TP - SWM & Decision Trees

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
import time

## A. SVM linéaire

In [ ]:
def afficher_frontiere(X, y, clf):

    
    decision_function = clf.decision_function(X)
    
    
    support_vector_indices = np.where(np.abs(decision_function) <= 1 + 1e-15)[0]
    support_vectors = X[support_vector_indices]

    plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)
    ax = plt.gca()
    DecisionBoundaryDisplay.from_estimator(
        clf,
        X,
        ax=ax,
        grid_resolution=50,
        plot_method="contour",
        colors="k",
        levels=[-1, 0, 1],
        alpha=0.5,
        linestyles=["--", "-", "--"],
    )
    plt.scatter(
        support_vectors[:, 0],
        support_vectors[:, 1],
        s=100,
        linewidth=1,
        facecolors="none",
        edgecolors="k",
    )
    plt.title("frontiere et vecteur de support")
    plt.show()

    return len(support_vectors)

In [ ]:
def afficher_frontiere_arbre(X, y, clf):


    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 500),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 500))

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)

    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap=plt.cm.RdYlBu)

    plt.title("frontiere et vecteur de support")
    plt.show()

    

### 1) Généralisation des données

In [ ]:
X_blob, y_blob = make_blobs(n_samples=1000, n_features=2, centers=2, cluster_std=1.0,
center_box=(-10.0, 10.0), random_state=1)

In [ ]:
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob)
plt.show()


### Apprentissage

In [ ]:
lin_blob_SVM = LinearSVC(C=10000)
lin_blob_SVM.fit(X_blob, y_blob)

print("Score du SVM lineaire apres entrainement sur toute la base de donnees : ")
print(f'{lin_blob_SVM.score(X_blob, y_blob) * 100} %')

afficher_frontiere(X_blob, y_blob, lin_blob_SVM)

b)  
Pour C = 10000,  
On constate qu'un seul vecteur de support est utilisé pour crée la frontière.  
La frontière délimite bien les deux datasets de manière strict.  

c)  
On obtient les mêmes résultats,   
La SVM est deterministe comparer au réseau de neuronne.  

Architecture du réseau neuroneaux :  
1 perceptron  

## B. SVM non linéaire à noyau gaussien

### 1) Généralisation des données

In [ ]:
X_moon, y_moon = make_moons(n_samples=1000, noise=0.2, random_state=1)

In [ ]:
plt.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon)
plt.show()

In [ ]:
X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(X_moon, y_moon, test_size=0.3, random_state=1)

In [ ]:
lin_moon_SVM = LinearSVC(C=10000)
lin_moon_SVM.fit(X_moon_train, y_moon_train)

print("Score du SVM lineaire apres entrainement sur toute la base de donnees : ")
print(f'{lin_moon_SVM.score(X_moon_test, y_moon_test) * 100} %')

afficher_frontiere(X_moon_train, y_moon_train, lin_moon_SVM)

### 2) Analyser l'influence du paramètre de dispersion (spread) y (gamma) du noyau

In [ ]:
gamma = [0.001, 0.01, 0.1, 1, 10, 100]
score_test_gauss_SVM = []
score_train_gauss_SVM = []
nb_support_vector = []

for g in gamma:
    gauss_moon_SVM = SVC(C=1, gamma=g, random_state=1).fit(X_moon_train, y_moon_train)

    score_test_gauss_SVM.append(gauss_moon_SVM.score(X_moon_test, y_moon_test))
    score_train_gauss_SVM.append(gauss_moon_SVM.score(X_moon_train, y_moon_train))

    nb_sp = afficher_frontiere(X_moon_train, y_moon_train, gauss_moon_SVM)
    nb_support_vector.append(nb_sp)


In [ ]:
print([(1-i)*100 for i in score_train_gauss_SVM])
print([(1-i)*100 for i in score_test_gauss_SVM])
print(nb_support_vector)

In [ ]:
plt.plot(gamma, score_train_gauss_SVM, label='apprentissage')
plt.plot(gamma, score_test_gauss_SVM, label='test')
plt.xlabel('gamma')
plt.ylabel('precision')
plt.title('precision de la SVM en fonction de gamma')
plt.legend()
plt.show()

plt.plot(gamma, nb_support_vector)
plt.xlabel('gamma')
plt.ylabel('nombre de vecteur support')
plt.title('nombre de vecteur support en fonction de gamma')
plt.show()


### 3) Optimisation par recherche en grille (grid search)

In [ ]:
param_grid = {
    'C' : [0.1, 1, 10, 100],
    'gamma' : [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(SVC(), param_grid=param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_moon_train, y_moon_train)

best_model = grid_search.best_estimator_

In [ ]:
print(best_model)

In [ ]:
print('précision du model : ')
print(f'{best_model.score(X_moon_test, y_moon_test) * 100} %')

In [ ]:
nb_sp_opt = afficher_frontiere(X_moon_train, y_moon_train, best_model)
print(f'nombre de support de vecteur optimaux : {nb_sp_opt}')

temps d'inférence ???

## Decision trees

### 1) Analyer l'impact des paramètres (critères d'arrêt de l'apprentissage)

In [ ]:
profondeur = [i for i in range(1, 100, 1)]

score_test_arbre = []
score_train_arbre = []

for i in profondeur:
    arbre = DecisionTreeClassifier(criterion='entropy', max_depth=i)
    arbre.fit(X_moon_train, y_moon_train)

    # afficher_frontiere_arbre(X_moon_train, y_moon_train, arbre)

    score_test_arbre.append(arbre.score(X_moon_test, y_moon_test))
    score_train_arbre.append(arbre.score(X_moon_train, y_moon_train))

In [ ]:
plt.plot(profondeur, score_test_arbre, label='test')
plt.plot(profondeur, score_train_arbre, label='apprentissage')
plt.legend()
plt.ylabel('precision')
plt.xlabel('max_depth')
plt.title('impact de max_depth sur la precision d\'un arbre')
plt.show()

In [ ]:
sample = [i for i in range(1, 100)]

score_test_arbre_sample = []
score_train_arbre_sample = []

profondeur_arbre = []

for i in sample:
    arbre_sample = DecisionTreeClassifier(criterion='entropy', max_depth=None, min_samples_split=i)
    arbre_sample.fit(X_moon_train, y_moon_train)

    afficher_frontiere_arbre(X_moon_train, y_moon_train, arbre_sample)

    score_test_arbre_sample.append(arbre_sample.score(X_moon_test, y_moon_test))
    score_train_arbre_sample.append(arbre_sample.score(X_moon_train, y_moon_train))

    profondeur_arbre.append(arbre_sample.tree_.max_depth)

In [ ]:
plt.plot(sample, score_test_arbre_sample, label='test')
plt.plot(sample, score_train_arbre_sample, label='apprentissage')
plt.xlabel('min_sample_split')
plt.ylabel('precision')
plt.title('impact de min_sample_split sur la precision d\'un arbre')
plt.legend()
plt.show()

plt.plot(sample, profondeur_arbre)
plt.xlabel('min_sample_split')
plt.ylabel('profondeur')
plt.title('profondeur de l\'arbre en fonction de min_sample_split')
plt.show()

### 2) Optimisation par recherche en grille (grid search)

In [ ]:
param_grid_arbre = {
    'min_samples_split' : [i for i in range(1, 100)],
    'max_depth' : [i for i in range(1, 15)]
}

grid_search_arbre = GridSearchCV(DecisionTreeClassifier(criterion='entropy'), param_grid=param_grid_arbre, cv=5, scoring='accuracy')

grid_search_arbre.fit(X_moon_train, y_moon_train)

best_model_arbre = grid_search_arbre.best_estimator_

In [ ]:
print(best_model_arbre)
debut = time.time()
print(grid_search_arbre.score(X_moon_test, y_moon_test))
ecart = time.time() - debut
print(ecart)

In [ ]:
print(grid_search_arbre.cv_results_)

In [ ]:
afficher_frontiere_arbre(X_moon_train, y_moon_train, grid_search_arbre)

# Extra trees

In [ ]:
score_test_ex_tree = []
score_train_ex_tree = []

for i in range(20):
    ex_tree = ExtraTreeClassifier(splitter='random')
    ex_tree.fit(X_moon_train, y_moon_train)

    score_test_ex_tree.append(ex_tree.score(X_moon_test, y_moon_test))
    score_train_ex_tree.append(ex_tree.score(X_moon_train, y_moon_train))

In [ ]:
plt.boxplot(score_test_ex_tree, positions=[1])
plt.boxplot(score_train_ex_tree, positions=[2])
plt.gca().xaxis.set_ticklabels(['test', 'apprentissage'])
plt.title('boite à moustache des scores sur le extra tree')
plt.ylabel('precision')
plt.plot()

In [ ]:
appr = [20, 50, 100]

score_train_ex_trees = []
score_test_ex_trees = []
temps_inference = []

for i in appr:
    ex_trees = ExtraTreesClassifier(n_estimators=i)
    ex_trees.fit(X_moon_train, y_moon_train)

    score_train_ex_trees.append(ex_trees.score(X_moon_train, y_moon_train))
    debut = time.time()
    score_test_ex_trees.append(ex_trees.score(X_moon_test, y_moon_test))
    ecart = time.time() - debut

    temps_inference.append(ecart)






In [ ]:
print(score_test_ex_trees)
print(temps_inference)